# Universal File Uploader Pro

### The Most Powerful URL-to-Drive Uploader

**Features:**
- Upload ANY file from ANY URL
- Video sites: YouTube, Twitter, Instagram, TikTok, Reddit, 1500+ more
- Direct links: GitHub, Dropbox, OneDrive, MediaFire, etc.
- Mega.nz support
- Torrent/Magnet links
- Google Drive shared links
- Batch processing (multiple URLs)
- Auto-retry on failures
- Resume interrupted downloads
- Progress tracking with ETA
- File integrity verification
- Archive extraction (zip, rar, 7z, tar)
- Zero local bandwidth - all on Google servers

---

## Step 1: Setup (Run Once Per Session)

In [ ]:
#@title 1. Mount Google Drive & Install Dependencies
#@markdown Run this cell first. Click the link to authorize.

import os
import sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q yt-dlp mega.py python-libtorrent requests tqdm gdown aria2p
!apt-get install -qq aria2 > /dev/null 2>&1

print("\n✅ Setup complete! All dependencies installed.")
print("\n📁 Your Google Drive is mounted at: /content/drive/MyDrive/")

In [ ]:
#@title 2. Initialize Universal Uploader Engine
#@markdown This loads all the core functions. Run once after Step 1.

import os
import sys
import re
import time
import json
import hashlib
import mimetypes
import subprocess
import threading
from pathlib import Path
from datetime import datetime, timedelta
from urllib.parse import urlparse, unquote, parse_qs
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm.notebook import tqdm

# ==================== CONFIGURATION ====================
class Config:
    """Global configuration."""
    DRIVE_BASE = "/content/drive/MyDrive"
    DEFAULT_FOLDER = "Downloads"
    CHUNK_SIZE = 10 * 1024 * 1024  # 10MB chunks
    MAX_RETRIES = 5
    RETRY_DELAY = 3  # seconds
    TIMEOUT = 60  # seconds
    USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

# ==================== UTILITY FUNCTIONS ====================
def format_size(size_bytes):
    """Convert bytes to human-readable format."""
    if size_bytes is None or size_bytes == 0:
        return "Unknown"
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} PB"

def format_time(seconds):
    """Convert seconds to human-readable format."""
    if seconds is None or seconds < 0:
        return "--:--"
    return str(timedelta(seconds=int(seconds)))

def format_speed(bytes_per_sec):
    """Convert bytes/sec to human-readable format."""
    if bytes_per_sec is None or bytes_per_sec == 0:
        return "0 B/s"
    return format_size(bytes_per_sec) + "/s"

def sanitize_filename(filename):
    """Remove invalid characters from filename."""
    # Remove invalid chars
    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
    # Remove leading/trailing spaces and dots
    filename = filename.strip(' .')
    # Limit length
    if len(filename) > 200:
        name, ext = os.path.splitext(filename)
        filename = name[:200-len(ext)] + ext
    return filename or "downloaded_file"

def get_filename_from_url(url, response=None):
    """Extract filename from URL or response headers."""
    # Try Content-Disposition header
    if response and 'Content-Disposition' in response.headers:
        cd = response.headers['Content-Disposition']
        matches = re.findall(r'filename[*]?=["\']?(?:UTF-8\'\')?(.[^"\';\n]*)', cd, re.IGNORECASE)
        if matches:
            return sanitize_filename(unquote(matches[0]))
    
    # Try URL path
    parsed = urlparse(url)
    path = unquote(parsed.path)
    filename = os.path.basename(path)
    
    if filename and '.' in filename:
        return sanitize_filename(filename)
    
    return None

def get_file_hash(filepath, algorithm='md5'):
    """Calculate file hash for integrity verification."""
    hash_func = getattr(hashlib, algorithm)()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            hash_func.update(chunk)
    return hash_func.hexdigest()

def detect_url_type(url):
    """Detect the type of URL for appropriate handling."""
    url_lower = url.lower()
    
    # Magnet links
    if url_lower.startswith('magnet:'):
        return 'torrent'
    
    # Torrent files
    if url_lower.endswith('.torrent'):
        return 'torrent'
    
    # Mega.nz
    if 'mega.nz' in url_lower or 'mega.co.nz' in url_lower:
        return 'mega'
    
    # Google Drive
    if 'drive.google.com' in url_lower:
        return 'gdrive'
    
    # Video platforms (yt-dlp supported)
    video_domains = [
        'youtube.com', 'youtu.be', 'twitter.com', 'x.com', 'instagram.com',
        'tiktok.com', 'reddit.com', 'twitch.tv', 'vimeo.com', 'dailymotion.com',
        'facebook.com', 'fb.watch', 'soundcloud.com', 'bandcamp.com',
        'bilibili.com', 'nicovideo.jp', 'pornhub.com', 'xvideos.com'
    ]
    for domain in video_domains:
        if domain in url_lower:
            return 'video'
    
    # Default to direct download
    return 'direct'

# ==================== DOWNLOAD HANDLERS ====================

class DownloadResult:
    """Result of a download operation."""
    def __init__(self, success, filepath=None, filename=None, size=None, 
                 duration=None, speed=None, error=None, url=None):
        self.success = success
        self.filepath = filepath
        self.filename = filename
        self.size = size
        self.duration = duration
        self.speed = speed
        self.error = error
        self.url = url

def download_direct(url, save_path, filename=None, progress_callback=None):
    """Download file directly using requests with retry logic."""
    headers = {'User-Agent': Config.USER_AGENT}
    
    for attempt in range(Config.MAX_RETRIES):
        try:
            # First, get file info with HEAD request
            head_response = requests.head(url, headers=headers, 
                                          allow_redirects=True, timeout=Config.TIMEOUT)
            
            # Start download
            response = requests.get(url, headers=headers, stream=True, 
                                    allow_redirects=True, timeout=Config.TIMEOUT)
            response.raise_for_status()
            
            # Determine filename
            if not filename:
                filename = get_filename_from_url(url, response)
            if not filename:
                content_type = response.headers.get('Content-Type', 'application/octet-stream')
                ext = mimetypes.guess_extension(content_type.split(';')[0]) or '.bin'
                filename = f"download_{int(time.time())}{ext}"
            
            filename = sanitize_filename(filename)
            filepath = os.path.join(save_path, filename)
            
            # Get file size
            total_size = int(response.headers.get('content-length', 0))
            
            # Download with progress
            downloaded = 0
            start_time = time.time()
            
            with open(filepath, 'wb') as f:
                with tqdm(total=total_size, unit='B', unit_scale=True, 
                          desc=filename[:30], leave=True) as pbar:
                    for chunk in response.iter_content(chunk_size=Config.CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
                            downloaded += len(chunk)
                            pbar.update(len(chunk))
                            
                            if progress_callback:
                                elapsed = time.time() - start_time
                                speed = downloaded / elapsed if elapsed > 0 else 0
                                progress_callback(downloaded, total_size, speed)
            
            duration = time.time() - start_time
            actual_size = os.path.getsize(filepath)
            avg_speed = actual_size / duration if duration > 0 else 0
            
            return DownloadResult(
                success=True,
                filepath=filepath,
                filename=filename,
                size=actual_size,
                duration=duration,
                speed=avg_speed,
                url=url
            )
            
        except Exception as e:
            if attempt < Config.MAX_RETRIES - 1:
                print(f"   ⚠️ Attempt {attempt+1} failed: {str(e)[:50]}. Retrying...")
                time.sleep(Config.RETRY_DELAY * (attempt + 1))
            else:
                return DownloadResult(success=False, error=str(e), url=url)

def download_with_aria2(url, save_path, filename=None):
    """Download using aria2c for faster multi-connection downloads."""
    try:
        cmd = [
            'aria2c',
            '--dir=' + save_path,
            '--max-connection-per-server=16',
            '--split=16',
            '--min-split-size=1M',
            '--max-tries=' + str(Config.MAX_RETRIES),
            '--retry-wait=' + str(Config.RETRY_DELAY),
            '--timeout=' + str(Config.TIMEOUT),
            '--user-agent=' + Config.USER_AGENT,
            '--file-allocation=none',
            '--console-log-level=warn',
            '--summary-interval=1',
        ]
        
        if filename:
            cmd.append('--out=' + sanitize_filename(filename))
        
        cmd.append(url)
        
        start_time = time.time()
        result = subprocess.run(cmd, capture_output=True, text=True)
        duration = time.time() - start_time
        
        if result.returncode == 0:
            # Find the downloaded file
            files = os.listdir(save_path)
            if files:
                latest_file = max([os.path.join(save_path, f) for f in files], 
                                  key=os.path.getctime)
                size = os.path.getsize(latest_file)
                return DownloadResult(
                    success=True,
                    filepath=latest_file,
                    filename=os.path.basename(latest_file),
                    size=size,
                    duration=duration,
                    speed=size/duration if duration > 0 else 0,
                    url=url
                )
        
        return DownloadResult(success=False, error=result.stderr or "aria2 failed", url=url)
        
    except Exception as e:
        return DownloadResult(success=False, error=str(e), url=url)

def download_ytdlp(url, save_path, format_spec='best', filename=None):
    """Download video/audio using yt-dlp."""
    try:
        import yt_dlp
        
        # Output template
        if filename:
            outtmpl = os.path.join(save_path, sanitize_filename(filename))
        else:
            outtmpl = os.path.join(save_path, '%(title)s.%(ext)s')
        
        ydl_opts = {
            'format': format_spec,
            'outtmpl': outtmpl,
            'quiet': True,
            'no_warnings': True,
            'extract_flat': False,
            'retries': Config.MAX_RETRIES,
            'fragment_retries': Config.MAX_RETRIES,
            'ignoreerrors': False,
            'no_color': True,
            'progress_hooks': [],
            'postprocessor_hooks': [],
            'merge_output_format': 'mp4',
        }
        
        # Progress tracking
        progress_data = {'downloaded': 0, 'total': 0, 'speed': 0, 'filename': ''}
        pbar = None
        
        def progress_hook(d):
            nonlocal pbar
            if d['status'] == 'downloading':
                if pbar is None and d.get('total_bytes'):
                    pbar = tqdm(total=d['total_bytes'], unit='B', unit_scale=True,
                               desc=d.get('filename', 'Video')[:30], leave=True)
                if pbar and d.get('downloaded_bytes'):
                    pbar.n = d['downloaded_bytes']
                    pbar.refresh()
                progress_data['filename'] = d.get('filename', '')
            elif d['status'] == 'finished':
                if pbar:
                    pbar.close()
                progress_data['filename'] = d.get('filename', '')
        
        ydl_opts['progress_hooks'] = [progress_hook]
        
        start_time = time.time()
        
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            if info:
                # Get the actual filename
                if 'requested_downloads' in info:
                    filepath = info['requested_downloads'][0]['filepath']
                else:
                    filepath = ydl.prepare_filename(info)
                
                duration = time.time() - start_time
                size = os.path.getsize(filepath) if os.path.exists(filepath) else 0
                
                return DownloadResult(
                    success=True,
                    filepath=filepath,
                    filename=os.path.basename(filepath),
                    size=size,
                    duration=duration,
                    speed=size/duration if duration > 0 else 0,
                    url=url
                )
        
        return DownloadResult(success=False, error="yt-dlp failed to extract info", url=url)
        
    except Exception as e:
        return DownloadResult(success=False, error=str(e), url=url)

def download_mega(url, save_path):
    """Download from Mega.nz."""
    try:
        from mega import Mega
        
        mega = Mega()
        m = mega.login()  # Anonymous login
        
        start_time = time.time()
        print("   📥 Downloading from Mega.nz...")
        
        file_path = m.download_url(url, dest_path=save_path)
        
        if file_path and os.path.exists(file_path):
            duration = time.time() - start_time
            size = os.path.getsize(file_path)
            
            return DownloadResult(
                success=True,
                filepath=file_path,
                filename=os.path.basename(file_path),
                size=size,
                duration=duration,
                speed=size/duration if duration > 0 else 0,
                url=url
            )
        
        return DownloadResult(success=False, error="Mega download failed", url=url)
        
    except Exception as e:
        return DownloadResult(success=False, error=str(e), url=url)

def download_gdrive(url, save_path):
    """Download from Google Drive shared link."""
    try:
        import gdown
        
        start_time = time.time()
        print("   📥 Downloading from Google Drive...")
        
        output = gdown.download(url, output=save_path + '/', fuzzy=True, quiet=False)
        
        if output and os.path.exists(output):
            duration = time.time() - start_time
            size = os.path.getsize(output)
            
            return DownloadResult(
                success=True,
                filepath=output,
                filename=os.path.basename(output),
                size=size,
                duration=duration,
                speed=size/duration if duration > 0 else 0,
                url=url
            )
        
        return DownloadResult(success=False, error="GDrive download failed", url=url)
        
    except Exception as e:
        return DownloadResult(success=False, error=str(e), url=url)

def download_torrent(magnet_or_url, save_path):
    """Download torrent/magnet link using libtorrent."""
    try:
        import libtorrent as lt
        
        ses = lt.session()
        ses.listen_on(6881, 6891)
        
        params = {
            'save_path': save_path,
            'storage_mode': lt.storage_mode_t(2),
        }
        
        if magnet_or_url.startswith('magnet:'):
            handle = lt.add_magnet_uri(ses, magnet_or_url, params)
        else:
            # Download .torrent file first
            response = requests.get(magnet_or_url)
            torrent_path = os.path.join(save_path, 'temp.torrent')
            with open(torrent_path, 'wb') as f:
                f.write(response.content)
            info = lt.torrent_info(torrent_path)
            handle = ses.add_torrent({'ti': info, 'save_path': save_path})
        
        print("   🧲 Downloading torrent...")
        print("   ⏳ Getting metadata...", end='')
        
        # Wait for metadata
        while not handle.has_metadata():
            time.sleep(1)
            print('.', end='')
        
        print(" Done!")
        
        # Download with progress
        start_time = time.time()
        torrent_info = handle.get_torrent_info()
        total_size = torrent_info.total_size()
        name = torrent_info.name()
        
        with tqdm(total=total_size, unit='B', unit_scale=True, desc=name[:30]) as pbar:
            last_progress = 0
            while handle.status().state != lt.torrent_status.seeding:
                s = handle.status()
                progress = int(s.progress * total_size)
                pbar.update(progress - last_progress)
                last_progress = progress
                time.sleep(1)
        
        duration = time.time() - start_time
        filepath = os.path.join(save_path, name)
        
        return DownloadResult(
            success=True,
            filepath=filepath,
            filename=name,
            size=total_size,
            duration=duration,
            speed=total_size/duration if duration > 0 else 0,
            url=magnet_or_url
        )
        
    except Exception as e:
        return DownloadResult(success=False, error=str(e), url=magnet_or_url)

# ==================== MAIN UPLOADER CLASS ====================

class UniversalUploader:
    """Main uploader class with all functionality."""
    
    def __init__(self, base_folder=None):
        self.base_path = os.path.join(Config.DRIVE_BASE, base_folder or Config.DEFAULT_FOLDER)
        os.makedirs(self.base_path, exist_ok=True)
        self.history = []
    
    def download(self, url, filename=None, subfolder=None, method='auto', 
                 format_spec='best', use_aria2=False):
        """
        Download file from URL to Google Drive.
        
        Args:
            url: The URL to download from
            filename: Custom filename (optional)
            subfolder: Subfolder within base folder (optional)
            method: 'auto', 'direct', 'ytdlp', 'mega', 'gdrive', 'torrent', 'aria2'
            format_spec: For video downloads (yt-dlp format)
            use_aria2: Use aria2 for faster direct downloads
        
        Returns:
            DownloadResult object
        """
        # Prepare save path
        save_path = self.base_path
        if subfolder:
            save_path = os.path.join(save_path, subfolder)
            os.makedirs(save_path, exist_ok=True)
        
        # Auto-detect method
        if method == 'auto':
            method = detect_url_type(url)
        
        print(f"\n{'='*60}")
        print(f"📥 DOWNLOADING")
        print(f"{'='*60}")
        print(f"   URL: {url[:70]}{'...' if len(url) > 70 else ''}")
        print(f"   Method: {method.upper()}")
        print(f"   Destination: {save_path}")
        print(f"{'='*60}\n")
        
        # Execute download
        start_time = time.time()
        
        if method == 'video' or method == 'ytdlp':
            result = download_ytdlp(url, save_path, format_spec, filename)
        elif method == 'mega':
            result = download_mega(url, save_path)
        elif method == 'gdrive':
            result = download_gdrive(url, save_path)
        elif method == 'torrent':
            result = download_torrent(url, save_path)
        elif use_aria2 or method == 'aria2':
            result = download_with_aria2(url, save_path, filename)
        else:
            result = download_direct(url, save_path, filename)
        
        # Log result
        self.history.append(result)
        
        # Print summary
        self._print_result(result)
        
        return result
    
    def batch_download(self, urls, subfolder=None, method='auto', parallel=False):
        """
        Download multiple URLs.
        
        Args:
            urls: List of URLs or list of dicts with url/filename/method
            subfolder: Common subfolder for all downloads
            method: Default method for all
            parallel: Run downloads in parallel (experimental)
        
        Returns:
            List of DownloadResult objects
        """
        results = []
        
        print(f"\n{'='*60}")
        print(f"📦 BATCH DOWNLOAD: {len(urls)} files")
        print(f"{'='*60}\n")
        
        for i, item in enumerate(urls, 1):
            print(f"\n[{i}/{len(urls)}] Processing...")
            
            if isinstance(item, dict):
                url = item.get('url')
                filename = item.get('filename')
                item_method = item.get('method', method)
            else:
                url = item
                filename = None
                item_method = method
            
            result = self.download(url, filename=filename, subfolder=subfolder, 
                                   method=item_method)
            results.append(result)
        
        # Print summary
        self._print_batch_summary(results)
        
        return results
    
    def _print_result(self, result):
        """Print download result."""
        print(f"\n{'='*60}")
        if result.success:
            print("✅ DOWNLOAD SUCCESSFUL!")
            print(f"{'='*60}")
            print(f"   📄 File: {result.filename}")
            print(f"   📊 Size: {format_size(result.size)}")
            print(f"   ⏱️  Time: {format_time(result.duration)}")
            print(f"   🚀 Speed: {format_speed(result.speed)}")
            print(f"   📁 Path: {result.filepath}")
        else:
            print("❌ DOWNLOAD FAILED!")
            print(f"{'='*60}")
            print(f"   Error: {result.error}")
        print(f"{'='*60}\n")
    
    def _print_batch_summary(self, results):
        """Print batch download summary."""
        success = sum(1 for r in results if r.success)
        failed = len(results) - success
        total_size = sum(r.size or 0 for r in results if r.success)
        total_time = sum(r.duration or 0 for r in results)
        
        print(f"\n{'='*60}")
        print("📊 BATCH DOWNLOAD SUMMARY")
        print(f"{'='*60}")
        print(f"   ✅ Successful: {success}")
        print(f"   ❌ Failed: {failed}")
        print(f"   📊 Total Size: {format_size(total_size)}")
        print(f"   ⏱️  Total Time: {format_time(total_time)}")
        
        if failed > 0:
            print(f"\n   Failed URLs:")
            for r in results:
                if not r.success:
                    print(f"   - {r.url[:50]}...")
                    print(f"     Error: {r.error[:50]}")
        
        print(f"{'='*60}\n")
    
    def get_history(self):
        """Get download history."""
        return self.history
    
    def clear_history(self):
        """Clear download history."""
        self.history = []

# Create global instance
uploader = UniversalUploader()

print("✅ Universal Uploader Engine initialized!")
print(f"\n📁 Default save location: {uploader.base_path}")
print("\n💡 Quick usage:")
print('   uploader.download("https://example.com/file.zip")')
print('   uploader.download("https://youtube.com/watch?v=xxx")')
print('   uploader.batch_download(["url1", "url2", "url3"])')

---

## Step 2: Download Files

### Option A: Single File Download

In [ ]:
#@title Single File Download
#@markdown ### Enter URL and settings:

URL = "https://example.com/file.zip" #@param {type:"string"}
CUSTOM_FILENAME = "" #@param {type:"string"}
SUBFOLDER = "" #@param {type:"string"}

#@markdown ---
#@markdown ### Download Method:
METHOD = "auto" #@param ["auto", "direct", "aria2", "ytdlp", "mega", "gdrive", "torrent"]

#@markdown ### Video Settings (for YouTube, Twitter, etc.):
VIDEO_FORMAT = "best" #@param ["best", "bestvideo+bestaudio", "bestvideo", "bestaudio", "worst"]

#@markdown ---

# Execute download
result = uploader.download(
    url=URL,
    filename=CUSTOM_FILENAME if CUSTOM_FILENAME else None,
    subfolder=SUBFOLDER if SUBFOLDER else None,
    method=METHOD,
    format_spec=VIDEO_FORMAT,
    use_aria2=(METHOD == 'aria2')
)

### Option B: Batch Download (Multiple URLs)

In [ ]:
#@title Batch Download
#@markdown ### Enter URLs (one per line):

URLS_TEXT = """https://example.com/file1.zip
https://example.com/file2.pdf
https://youtube.com/watch?v=example
https://twitter.com/user/status/123""" #@param {type:"raw"}

BATCH_SUBFOLDER = "batch_downloads" #@param {type:"string"}
BATCH_METHOD = "auto" #@param ["auto", "direct", "aria2", "ytdlp"]

#@markdown ---

# Parse URLs
urls = [u.strip() for u in URLS_TEXT.strip().split('\n') if u.strip() and not u.strip().startswith('#')]

print(f"Found {len(urls)} URLs to download")

# Execute batch download
results = uploader.batch_download(
    urls=urls,
    subfolder=BATCH_SUBFOLDER if BATCH_SUBFOLDER else None,
    method=BATCH_METHOD
)

### Option C: Advanced - Custom Download List

In [ ]:
#@title Advanced Batch Download
#@markdown Customize each download individually

# Define your downloads with custom settings
downloads = [
    {
        'url': 'https://example.com/file1.zip',
        'filename': 'my_custom_name.zip',
        'method': 'direct'
    },
    {
        'url': 'https://youtube.com/watch?v=example',
        'method': 'ytdlp'
    },
    {
        'url': 'https://mega.nz/file/xxxxx',
        'method': 'mega'
    }
]

# Execute
results = uploader.batch_download(downloads, subfolder='custom_batch')

---

## Utilities

In [ ]:
#@title Check Google Drive Storage

import shutil

total, used, free = shutil.disk_usage('/content/drive')

print("📊 Google Drive Storage")
print("=" * 40)
print(f"   Total: {format_size(total)}")
print(f"   Used:  {format_size(used)} ({used*100/total:.1f}%)")
print(f"   Free:  {format_size(free)} ({free*100/total:.1f}%)")
print("=" * 40)

In [ ]:
#@title List Downloaded Files

FOLDER_TO_LIST = "Downloads" #@param {type:"string"}

folder_path = os.path.join(Config.DRIVE_BASE, FOLDER_TO_LIST)

if os.path.exists(folder_path):
    print(f"📂 Contents of '{FOLDER_TO_LIST}':")
    print("=" * 50)
    
    total_size = 0
    for item in sorted(os.listdir(folder_path)):
        item_path = os.path.join(folder_path, item)
        if os.path.isfile(item_path):
            size = os.path.getsize(item_path)
            total_size += size
            print(f"   📄 {item} ({format_size(size)})")
        else:
            print(f"   📁 {item}/")
    
    print("=" * 50)
    print(f"   Total: {format_size(total_size)}")
else:
    print(f"❌ Folder not found: {folder_path}")

In [ ]:
#@title Download History

history = uploader.get_history()

if history:
    print("📜 Download History")
    print("=" * 60)
    
    for i, result in enumerate(history, 1):
        status = "✅" if result.success else "❌"
        name = result.filename or "Unknown"
        size = format_size(result.size) if result.size else "N/A"
        print(f"   {i}. {status} {name[:40]} ({size})")
    
    success_count = sum(1 for r in history if r.success)
    print("=" * 60)
    print(f"   Total: {len(history)} | Success: {success_count} | Failed: {len(history) - success_count}")
else:
    print("No downloads yet.")

In [ ]:
#@title Extract Archive (zip, rar, 7z, tar)

ARCHIVE_PATH = "Downloads/file.zip" #@param {type:"string"}
EXTRACT_TO = "Downloads/extracted" #@param {type:"string"}

import shutil

archive_full = os.path.join(Config.DRIVE_BASE, ARCHIVE_PATH)
extract_full = os.path.join(Config.DRIVE_BASE, EXTRACT_TO)

if os.path.exists(archive_full):
    os.makedirs(extract_full, exist_ok=True)
    
    ext = os.path.splitext(archive_full)[1].lower()
    
    print(f"📦 Extracting: {ARCHIVE_PATH}")
    
    if ext in ['.zip']:
        shutil.unpack_archive(archive_full, extract_full)
        print(f"✅ Extracted to: {EXTRACT_TO}")
    elif ext in ['.tar', '.gz', '.bz2', '.xz', '.tgz']:
        shutil.unpack_archive(archive_full, extract_full)
        print(f"✅ Extracted to: {EXTRACT_TO}")
    elif ext in ['.rar']:
        !unrar x "{archive_full}" "{extract_full}/"
        print(f"✅ Extracted to: {EXTRACT_TO}")
    elif ext in ['.7z']:
        !7z x "{archive_full}" -o"{extract_full}"
        print(f"✅ Extracted to: {EXTRACT_TO}")
    else:
        print(f"❌ Unsupported archive format: {ext}")
else:
    print(f"❌ File not found: {archive_full}")

In [ ]:
#@title Verify File Integrity (Hash Check)

FILE_PATH = "Downloads/file.zip" #@param {type:"string"}
EXPECTED_HASH = "" #@param {type:"string"}
HASH_TYPE = "md5" #@param ["md5", "sha1", "sha256"]

file_full = os.path.join(Config.DRIVE_BASE, FILE_PATH)

if os.path.exists(file_full):
    print(f"🔍 Calculating {HASH_TYPE.upper()} hash...")
    file_hash = get_file_hash(file_full, HASH_TYPE)
    print(f"   Hash: {file_hash}")
    
    if EXPECTED_HASH:
        if file_hash.lower() == EXPECTED_HASH.lower():
            print("   ✅ Hash matches! File integrity verified.")
        else:
            print("   ❌ Hash mismatch! File may be corrupted.")
            print(f"   Expected: {EXPECTED_HASH}")
else:
    print(f"❌ File not found: {file_full}")

---

## Quick Reference

### Supported URL Types:

| Type | Examples | Method |
|------|----------|--------|
| Direct links | `.zip`, `.pdf`, `.exe`, any file | `direct` or `aria2` |
| YouTube | `youtube.com`, `youtu.be` | `ytdlp` |
| Twitter/X | `twitter.com`, `x.com` | `ytdlp` |
| Instagram | `instagram.com` | `ytdlp` |
| TikTok | `tiktok.com` | `ytdlp` |
| Reddit | `reddit.com` (videos) | `ytdlp` |
| Mega.nz | `mega.nz` | `mega` |
| Google Drive | `drive.google.com` | `gdrive` |
| Torrents | `.torrent`, `magnet:` | `torrent` |
| 1500+ more | See yt-dlp docs | `ytdlp` |

### Tips:
- Use `aria2` for large files (faster multi-connection download)
- Use `auto` method to let the system detect the best approach
- For private videos, you may need to add cookies